# 拡張GraphRAG vs 構造化RAG: 比較評価（90テストケース）

拡張されたGraphRAG（新関係性エッジ追加）と構造化RAGを比較評価します。

## 拡張GraphRAGの新機能
- **SAME_BRAND**: 同一ブランド/チェーン関係
- **COMPLEMENTARY**: 補完関係（ホテル↔レストラン等）
- **COMPETITOR**: 競合関係（同カテゴリ近接POI）
- **SAME_CUISINE**: 同一料理ジャンル関係
- **SAME_HOURS**: 営業時間類似関係（24時間/深夜/早朝）

## テストケース構成
- **55件**: 構造化RAG向け（L1-L5）
- **35件**: 拡張GraphRAG向け（GR-01〜GR-35）
- **合計90件**

## 1. 環境セットアップ

In [ ]:
# Google Colab環境チェック
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # リポジトリのクローン
    !git clone https://github.com/mopinfish/experiments-local-llm.git /content/repo
    %cd /content/repo
    !git checkout feature/graphrag-experiment
    !git pull origin feature/graphrag-experiment
    
    # 依存関係インストール
    !pip install -q transformers accelerate bitsandbytes torch networkx chromadb sentence-transformers
    !pip install -q langchain langchain-community
    !pip install -q japanize_matplotlib
    
    # srcをパスに追加
    sys.path.insert(0, '/content/repo/src')
    REPO_ROOT = '/content/repo'
else:
    project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    sys.path.insert(0, os.path.join(project_root, 'src'))
    REPO_ROOT = project_root

print(f"Running in: {'Google Colab' if IN_COLAB else 'Local environment'}")

In [ ]:
# 共通インポート
import json
import time
from datetime import datetime
from typing import Dict, List, Any, Tuple
from dataclasses import dataclass, asdict
from collections import defaultdict

import matplotlib.pyplot as plt
import japanize_matplotlib
import pandas as pd
import numpy as np

# テストケースのインポート
from test_cases_v2 import TEST_CASES_V2, get_test_case_stats_v2
from test_cases_graphrag import GRAPHRAG_TEST_CASES, get_graphrag_test_case_stats

# 統計表示
stats_v2 = get_test_case_stats_v2()
stats_gr = get_graphrag_test_case_stats()
print(f"構造化RAG向けテストケース: {stats_v2['total']}件")
print(f"拡張GraphRAG向けテストケース: {stats_gr['total']}件")
print(f"合計: {stats_v2['total'] + stats_gr['total']}件")
print(f"\nGraphRAGカテゴリ別:")
for cat, count in sorted(stats_gr['by_category'].items()):
    print(f"  {cat}: {count}件")

## 2. LLMの初期化

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4bit量子化設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("LLM loaded successfully!")

## 3. 拡張GraphRAGシステムの初期化

In [ ]:
# POIデータの読み込み
poi_path = f'{REPO_ROOT}/poi_documents.json'
with open(poi_path, 'r', encoding='utf-8') as f:
    pois = json.load(f)

print(f"Loaded {len(pois)} POIs")

# 拡張メタデータの確認
brand_count = sum(1 for p in pois if p['metadata'].get('brand'))
cuisine_count = sum(1 for p in pois if p['metadata'].get('cuisine'))
hours_count = sum(1 for p in pois if p['metadata'].get('opening_hours'))
print(f"\n拡張メタデータ:")
print(f"  ブランド情報あり: {brand_count}件 ({brand_count/len(pois)*100:.1f}%)")
print(f"  料理ジャンル情報あり: {cuisine_count}件 ({cuisine_count/len(pois)*100:.1f}%)")
print(f"  営業時間情報あり: {hours_count}件 ({hours_count/len(pois)*100:.1f}%)")

In [ ]:
# 拡張GraphRAGシステムの初期化
from graph_builder import POIGraphBuilder
from graph_rag_system import GraphRAGSystem

print("Building enhanced knowledge graph...")
builder = POIGraphBuilder()
graph = builder.build_graph(pois, include_extended_edges=True, verbose=True)

# GraphRAGシステム初期化
graph_rag = GraphRAGSystem(graph)
print(f"\nEnhanced GraphRAG initialized")
print(f"  Nodes: {graph.number_of_nodes()}")
print(f"  Edges: {graph.number_of_edges()}")

In [ ]:
# 構造化RAGシステムの初期化
from structured_rag_system import StructuredRAGSystem
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# ChromaDBパスの設定
chroma_path = f'{REPO_ROOT}/chroma_db'

# 埋め込みモデルの読み込み
embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={'device': 'cuda'}
)

# ベクトルストアの読み込み
vectorstore = Chroma(
    persist_directory=chroma_path,
    embedding_function=embedding_model
)

# 構造化RAGシステム初期化
structured_rag = StructuredRAGSystem(
    model=model,
    tokenizer=tokenizer,
    vectorstore=vectorstore,
    all_pois=pois,
    debug=False
)
print(f"Structured RAG initialized with {len(pois)} POIs")

## 4. 評価関数の定義

In [ ]:
def generate_response(question: str, context: str) -> str:
    """LLMで回答を生成"""
    prompt = f"""あなたは渋谷エリアのPOI（Point of Interest）情報に詳しいアシスタントです。
以下のコンテキスト情報を使用して、ユーザーの質問に正確に答えてください。

【コンテキスト】
{context}

【質問】
{question}

【回答】"""
    
    messages = [
        {"role": "system", "content": "あなたは正確で簡潔な回答を提供するアシスタントです。"},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()


def calculate_keyword_score(response: str, expected_keywords: List[str]) -> float:
    """キーワードヒット率を計算"""
    if not expected_keywords:
        return 1.0
    hits = sum(1 for kw in expected_keywords if kw.lower() in response.lower())
    return hits / len(expected_keywords)

In [ ]:
def evaluate_with_graphrag(question: str, expected_keywords: List[str]) -> Dict[str, Any]:
    """拡張GraphRAGで評価"""
    start_time = time.time()
    
    # グラフクエリでコンテキスト取得
    query_start = time.time()
    query_result = graph_rag.query(question)
    query_time = time.time() - query_start
    
    context = query_result.context if hasattr(query_result, 'context') else ''
    metadata = query_result.metadata if hasattr(query_result, 'metadata') else {}
    
    # LLMで回答生成
    llm_start = time.time()
    response = generate_response(question, context)
    llm_time = time.time() - llm_start
    
    total_time = time.time() - start_time
    keyword_score = calculate_keyword_score(response, expected_keywords)
    
    return {
        'response': response,
        'context': context[:500] if context else '',
        'keyword_score': keyword_score,
        'query_time': query_time,
        'llm_time': llm_time,
        'total_time': total_time,
        'metadata': metadata
    }


def evaluate_with_structured_rag(question: str, expected_keywords: List[str]) -> Dict[str, Any]:
    """構造化RAGで評価"""
    start_time = time.time()
    
    # 構造化RAGでコンテキスト取得
    query_start = time.time()
    context = structured_rag.get_context(question)
    query_time = time.time() - query_start
    
    # LLMで回答生成
    llm_start = time.time()
    response = generate_response(question, context)
    llm_time = time.time() - llm_start
    
    total_time = time.time() - start_time
    keyword_score = calculate_keyword_score(response, expected_keywords)
    
    return {
        'response': response,
        'context': context[:500] if context else '',
        'keyword_score': keyword_score,
        'query_time': query_time,
        'llm_time': llm_time,
        'total_time': total_time
    }

## 5. 統合テストケースの準備

In [ ]:
@dataclass
class UnifiedTestCase:
    """統合テストケース"""
    id: str
    source: str  # 'structured' or 'graphrag'
    category: str
    subcategory: str
    question: str
    expected_keywords: List[str]
    difficulty: str = 'medium'

# 統合テストケースを作成
unified_test_cases = []

# 構造化RAGテストケース（55件）
for tc in TEST_CASES_V2:
    unified_test_cases.append(UnifiedTestCase(
        id=tc.id,
        source='structured',
        category=tc.category,
        subcategory=tc.subcategory,
        question=tc.prompt,
        expected_keywords=tc.expected_keywords,
        difficulty=tc.difficulty
    ))

# 拡張GraphRAGテストケース（35件）
for tc in GRAPHRAG_TEST_CASES:
    unified_test_cases.append(UnifiedTestCase(
        id=tc.id,
        source='graphrag',
        category=tc.category,
        subcategory=tc.category,  # GraphRAGはcategoryのみ
        question=tc.question,
        expected_keywords=tc.expected_keywords,
        difficulty='medium'
    ))

print(f"統合テストケース: {len(unified_test_cases)}件")
print(f"  - 構造化RAG向け: {sum(1 for tc in unified_test_cases if tc.source == 'structured')}件")
print(f"  - 拡張GraphRAG向け: {sum(1 for tc in unified_test_cases if tc.source == 'graphrag')}件")

## 6. 両システムでの評価実行

In [ ]:
# 拡張GraphRAGでの評価
print("="*60)
print("Enhanced GraphRAG Evaluation")
print("="*60)

graphrag_results = []
total_start = time.time()

for i, tc in enumerate(unified_test_cases):
    print(f"\n[{i+1}/{len(unified_test_cases)}] {tc.id}: {tc.question[:40]}...")
    
    try:
        result = evaluate_with_graphrag(tc.question, tc.expected_keywords)
        result.update({
            'test_id': tc.id,
            'source': tc.source,
            'category': tc.category,
            'subcategory': tc.subcategory,
            'question': tc.question,
            'expected_keywords': tc.expected_keywords,
            'system': 'graphrag'
        })
        graphrag_results.append(result)
        print(f"  -> Score: {result['keyword_score']*100:.1f}% | Time: {result['total_time']:.1f}s")
    except Exception as e:
        print(f"  -> ERROR: {str(e)}")
        graphrag_results.append({
            'test_id': tc.id,
            'source': tc.source,
            'category': tc.category,
            'subcategory': tc.subcategory,
            'question': tc.question,
            'expected_keywords': tc.expected_keywords,
            'system': 'graphrag',
            'error': str(e),
            'keyword_score': 0.0,
            'total_time': 0.0
        })

graphrag_elapsed = time.time() - total_start
print(f"\nEnhanced GraphRAG evaluation completed in {graphrag_elapsed/60:.1f} minutes")

In [ ]:
# 構造化RAGでの評価
print("="*60)
print("Structured RAG Evaluation")
print("="*60)

structured_results = []
total_start = time.time()

for i, tc in enumerate(unified_test_cases):
    print(f"\n[{i+1}/{len(unified_test_cases)}] {tc.id}: {tc.question[:40]}...")
    
    try:
        result = evaluate_with_structured_rag(tc.question, tc.expected_keywords)
        result.update({
            'test_id': tc.id,
            'source': tc.source,
            'category': tc.category,
            'subcategory': tc.subcategory,
            'question': tc.question,
            'expected_keywords': tc.expected_keywords,
            'system': 'structured'
        })
        structured_results.append(result)
        print(f"  -> Score: {result['keyword_score']*100:.1f}% | Time: {result['total_time']:.1f}s")
    except Exception as e:
        print(f"  -> ERROR: {str(e)}")
        structured_results.append({
            'test_id': tc.id,
            'source': tc.source,
            'category': tc.category,
            'subcategory': tc.subcategory,
            'question': tc.question,
            'expected_keywords': tc.expected_keywords,
            'system': 'structured',
            'error': str(e),
            'keyword_score': 0.0,
            'total_time': 0.0
        })

structured_elapsed = time.time() - total_start
print(f"\nStructured RAG evaluation completed in {structured_elapsed/60:.1f} minutes")

## 7. クエリタイプ別適性マップの作成

In [ ]:
# DataFrameに変換
df_graphrag = pd.DataFrame(graphrag_results)
df_structured = pd.DataFrame(structured_results)

# カテゴリ別スコアの計算
def calculate_category_scores(df):
    return df.groupby('category')['keyword_score'].agg(['mean', 'std', 'count']).round(3)

graphrag_by_cat = calculate_category_scores(df_graphrag)
structured_by_cat = calculate_category_scores(df_structured)

print("="*60)
print("カテゴリ別スコア比較")
print("="*60)
print(f"\n{'カテゴリ':<25} {'GraphRAG':>12} {'構造化RAG':>12} {'差分':>10} {'推奨':>10}")
print("-"*70)

aptitude_map = []
for cat in sorted(set(graphrag_by_cat.index) | set(structured_by_cat.index)):
    gr_score = graphrag_by_cat.loc[cat, 'mean'] * 100 if cat in graphrag_by_cat.index else 0
    st_score = structured_by_cat.loc[cat, 'mean'] * 100 if cat in structured_by_cat.index else 0
    diff = gr_score - st_score
    
    if diff > 5:
        recommended = 'GraphRAG'
    elif diff < -5:
        recommended = '構造化RAG'
    else:
        recommended = '同等'
    
    aptitude_map.append({
        'category': cat,
        'graphrag_score': gr_score,
        'structured_score': st_score,
        'diff': diff,
        'recommended': recommended
    })
    
    print(f"{cat:<25} {gr_score:>11.1f}% {st_score:>11.1f}% {diff:>+9.1f}% {recommended:>10}")

aptitude_df = pd.DataFrame(aptitude_map)

In [ ]:
# ソース別（構造化RAG向け vs GraphRAG向けテストケース）の比較
print("\n" + "="*60)
print("テストケースソース別スコア比較")
print("="*60)

for source in ['structured', 'graphrag']:
    source_label = '構造化RAG向けテスト（55件）' if source == 'structured' else 'GraphRAG向けテスト（35件）'
    print(f"\n【{source_label}】")
    
    gr_subset = df_graphrag[df_graphrag['source'] == source]
    st_subset = df_structured[df_structured['source'] == source]
    
    gr_avg = gr_subset['keyword_score'].mean() * 100
    st_avg = st_subset['keyword_score'].mean() * 100
    
    print(f"  拡張GraphRAG: {gr_avg:.1f}%")
    print(f"  構造化RAG: {st_avg:.1f}%")
    print(f"  差分: {gr_avg - st_avg:+.1f}%")
    
    if gr_avg > st_avg:
        print(f"  -> 拡張GraphRAGが優位")
    elif st_avg > gr_avg:
        print(f"  -> 構造化RAGが優位")
    else:
        print(f"  -> 同等")

In [ ]:
# 拡張エッジタイプ別の分析（GraphRAGテストのみ）
print("\n" + "="*60)
print("拡張エッジタイプ別スコア（GraphRAG向けテストのみ）")
print("="*60)

# 新しいカテゴリのみ抽出
new_categories = ['brand', 'complementary', 'competitor', 'cuisine', 'hours']

print(f"\n{'エッジタイプ':<20} {'GraphRAG':>12} {'構造化RAG':>12} {'差分':>10}")
print("-"*55)

for cat in new_categories:
    gr_subset = df_graphrag[(df_graphrag['source'] == 'graphrag') & (df_graphrag['category'] == cat)]
    st_subset = df_structured[(df_structured['source'] == 'graphrag') & (df_structured['category'] == cat)]
    
    if len(gr_subset) > 0:
        gr_avg = gr_subset['keyword_score'].mean() * 100
        st_avg = st_subset['keyword_score'].mean() * 100
        diff = gr_avg - st_avg
        print(f"{cat:<20} {gr_avg:>11.1f}% {st_avg:>11.1f}% {diff:>+9.1f}%")

## 8. 可視化

In [ ]:
# 出力ディレクトリ
results_dir = f'{REPO_ROOT}/results'
os.makedirs(results_dir, exist_ok=True)

# カテゴリ別比較グラフ
fig, ax = plt.subplots(figsize=(16, 8))

categories = aptitude_df['category'].tolist()
x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width/2, aptitude_df['graphrag_score'], width, 
               label='拡張GraphRAG', color='#2196F3', edgecolor='black')
bars2 = ax.bar(x + width/2, aptitude_df['structured_score'], width,
               label='構造化RAG', color='#FF9800', edgecolor='black')

ax.set_xlabel('クエリカテゴリ')
ax.set_ylabel('スコア (%)')
ax.set_title('拡張GraphRAG vs 構造化RAG: クエリカテゴリ別適性比較（90テストケース）')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=45, ha='right')
ax.set_ylim(0, 100)
ax.legend()
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)

# 推奨システムを色で表示
for i, row in aptitude_df.iterrows():
    if row['recommended'] == 'GraphRAG':
        ax.annotate('*', (i - width/2, row['graphrag_score'] + 2), ha='center', fontsize=12, color='blue')
    elif row['recommended'] == '構造化RAG':
        ax.annotate('*', (i + width/2, row['structured_score'] + 2), ha='center', fontsize=12, color='orange')

plt.tight_layout()
plt.savefig(f'{results_dir}/enhanced_comparison_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {results_dir}/enhanced_comparison_by_category.png")

In [ ]:
# ソース別比較グラフ
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, source in enumerate(['structured', 'graphrag']):
    ax = axes[idx]
    source_label = '構造化RAG向けテスト(55件)' if source == 'structured' else 'GraphRAG向けテスト(35件)'
    
    gr_subset = df_graphrag[df_graphrag['source'] == source]
    st_subset = df_structured[df_structured['source'] == source]
    
    scores = [gr_subset['keyword_score'].mean() * 100, st_subset['keyword_score'].mean() * 100]
    colors = ['#2196F3', '#FF9800']
    labels = ['拡張GraphRAG', '構造化RAG']
    
    bars = ax.bar(labels, scores, color=colors, edgecolor='black')
    ax.set_ylabel('スコア (%)')
    ax.set_title(f'{source_label}での比較')
    ax.set_ylim(0, 100)
    
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                f'{score:.1f}%', ha='center', fontsize=12)

plt.tight_layout()
plt.savefig(f'{results_dir}/enhanced_comparison_by_source.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {results_dir}/enhanced_comparison_by_source.png")

In [ ]:
# 拡張エッジタイプ別グラフ
fig, ax = plt.subplots(figsize=(10, 6))

new_categories = ['brand', 'complementary', 'competitor', 'cuisine', 'hours']
gr_scores = []
st_scores = []

for cat in new_categories:
    gr_subset = df_graphrag[(df_graphrag['source'] == 'graphrag') & (df_graphrag['category'] == cat)]
    st_subset = df_structured[(df_structured['source'] == 'graphrag') & (df_structured['category'] == cat)]
    
    gr_scores.append(gr_subset['keyword_score'].mean() * 100 if len(gr_subset) > 0 else 0)
    st_scores.append(st_subset['keyword_score'].mean() * 100 if len(st_subset) > 0 else 0)

x = np.arange(len(new_categories))
width = 0.35

bars1 = ax.bar(x - width/2, gr_scores, width, label='拡張GraphRAG', color='#2196F3', edgecolor='black')
bars2 = ax.bar(x + width/2, st_scores, width, label='構造化RAG', color='#FF9800', edgecolor='black')

ax.set_xlabel('拡張エッジタイプ')
ax.set_ylabel('スコア (%)')
ax.set_title('拡張エッジタイプ別スコア比較（GraphRAG向けテストのみ）')
ax.set_xticks(x)
ax.set_xticklabels(new_categories)
ax.set_ylim(0, 100)
ax.legend()

plt.tight_layout()
plt.savefig(f'{results_dir}/enhanced_edge_type_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {results_dir}/enhanced_edge_type_comparison.png")

## 9. 結果の保存

In [ ]:
# 結果をJSONで保存
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# グラフ統計を取得
graph_stats = builder.get_stats()

output = {
    'metadata': {
        'timestamp': timestamp,
        'total_test_cases': len(unified_test_cases),
        'structured_test_cases': sum(1 for tc in unified_test_cases if tc.source == 'structured'),
        'graphrag_test_cases': sum(1 for tc in unified_test_cases if tc.source == 'graphrag'),
        'model': 'Qwen2.5-7B-Instruct (4bit)',
        'graph_stats': graph_stats
    },
    'summary': {
        'graphrag': {
            'overall_score': float(df_graphrag['keyword_score'].mean() * 100),
            'avg_time': float(df_graphrag['total_time'].mean()),
            'on_structured_tests': float(df_graphrag[df_graphrag['source'] == 'structured']['keyword_score'].mean() * 100),
            'on_graphrag_tests': float(df_graphrag[df_graphrag['source'] == 'graphrag']['keyword_score'].mean() * 100)
        },
        'structured': {
            'overall_score': float(df_structured['keyword_score'].mean() * 100),
            'avg_time': float(df_structured['total_time'].mean()),
            'on_structured_tests': float(df_structured[df_structured['source'] == 'structured']['keyword_score'].mean() * 100),
            'on_graphrag_tests': float(df_structured[df_structured['source'] == 'graphrag']['keyword_score'].mean() * 100)
        }
    },
    'aptitude_map': aptitude_df.to_dict(orient='records'),
    'graphrag_results': graphrag_results,
    'structured_results': structured_results
}

output_path = f'{results_dir}/enhanced_comparison_{timestamp}.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2, default=str)

print(f"Results saved to: {output_path}")

## 10. 結論と適性マップ

In [ ]:
print("="*60)
print("拡張GraphRAG比較評価 - 結論")
print("="*60)

# 総合スコア
gr_overall = df_graphrag['keyword_score'].mean() * 100
st_overall = df_structured['keyword_score'].mean() * 100

print(f"\n【総合スコア（90テストケース）】")
print(f"  拡張GraphRAG: {gr_overall:.1f}%")
print(f"  構造化RAG: {st_overall:.1f}%")
print(f"  差分: {gr_overall - st_overall:+.1f}%")

# 拡張エッジの効果
print(f"\n【拡張エッジの効果（GraphRAG向けテスト35件）】")
gr_on_gr = df_graphrag[df_graphrag['source'] == 'graphrag']['keyword_score'].mean() * 100
st_on_gr = df_structured[df_structured['source'] == 'graphrag']['keyword_score'].mean() * 100
print(f"  拡張GraphRAG: {gr_on_gr:.1f}%")
print(f"  構造化RAG: {st_on_gr:.1f}%")
print(f"  差分: {gr_on_gr - st_on_gr:+.1f}%")

# 適性マップ
print(f"\n【クエリタイプ別適性マップ】")
print(f"\n{'クエリタイプ':<25} {'推奨システム':>15}")
print("-"*45)

graphrag_wins = []
structured_wins = []
ties = []

for _, row in aptitude_df.iterrows():
    print(f"{row['category']:<25} {row['recommended']:>15}")
    if row['recommended'] == 'GraphRAG':
        graphrag_wins.append(row['category'])
    elif row['recommended'] == '構造化RAG':
        structured_wins.append(row['category'])
    else:
        ties.append(row['category'])

print(f"\n【ハイブリッドRAG設計指針】")
print(f"\n拡張GraphRAGを使用すべきクエリ ({len(graphrag_wins)}カテゴリ):")
for cat in graphrag_wins:
    print(f"  - {cat}")

print(f"\n構造化RAGを使用すべきクエリ ({len(structured_wins)}カテゴリ):")
for cat in structured_wins:
    print(f"  - {cat}")

if ties:
    print(f"\nどちらでも可（同等性能） ({len(ties)}カテゴリ):")
    for cat in ties:
        print(f"  - {cat}")